In [8]:
FILE = 'scratch/Smaalenenes_1903_Excel_Correction.xlsx'   # <-- set to your Excel file name
OUT  = 'validation_report.csv'

In [9]:
# finding stuff to flag 
import pandas as pd, re, unicodedata
from collections import Counter

df = pd.read_excel(FILE, dtype=str).fillna('').reset_index(drop=True)
cols = list(df.columns)
SKIP = {'check'}
def has(*c): return all(x in cols for x in c)

issues, skipped = [], []
def flag(check, i, detail):
    r = df.iloc[i] if i is not None else {}
    issues.append({'check': check, 'excel_row': (i + 2) if i is not None else '',
                   'gaards_no': r.get('gaards_no', ''), 'brugs_no': r.get('brugs_no', ''),
                   'detail': detail})

NUMERIC   = {c for c in cols if c.startswith('gaards_no') or c.startswith('brugs_no') or c in ('mark', 'ore')}
text_cols = [c for c in cols if c not in NUMERIC and c not in SKIP]
scan_cols = [c for c in cols if c not in SKIP]

num = lambda c: pd.to_numeric(df[c], errors='coerce') if c in cols else None
g, b, mk, oe = num('gaards_no'), num('brugs_no'), num('mark'), num('ore')
if mk is not None: mk = mk.fillna(0)
if oe is not None: oe = oe.fillna(0)

if 'eier_bruker' in cols:
    eb = df['eier_bruker'].str.strip()
    is_total = eb.str.contains('overf', case=False, na=False)
    is_sogn  = eb.str.contains(r'\bsogn\b', case=False, na=False) & ~eb.str.contains(r'\d')
    is_data  = ~is_total & ~is_sogn
else:
    is_total = is_sogn = pd.Series(False, index=df.index)
    is_data  = pd.Series(True, index=df.index)
DITTO = {'do.', 'do', '„', ',,', '"'}

# structure: gaards steps by 1
if g is not None:
    seen = []
    for i in df.index[is_data]:
        if pd.notna(g[i]):
            gi = int(g[i])
            if not seen or gi != seen[-1]:
                if seen and gi != seen[-1] + 1: flag('structure', i, f'gaards {seen[-1]} -> {gi}')
                seen.append(gi)
else: skipped.append('structure/gaards (no gaards_no)')

# structure: brugs resets to 1 and increments within each gaards
if g is not None and b is not None:
    gfill = g.where(g.notna()).ffill()
    cur, exp = None, 1
    for i in df.index[is_data]:
        if pd.isna(gfill[i]): continue
        if gfill[i] != cur: cur, exp = gfill[i], 1
        if pd.notna(b[i]):
            if int(b[i]) != exp: flag('structure', i, f'brugs gaards {int(cur)}: {int(b[i])} != {exp}')
            exp = int(b[i]) + 1
        else: exp += 1
else: skipped.append('structure/brugs (no gaards_no or brugs_no)')

# type safety: numeric columns clean
num_re, ore_re = re.compile(r'^\d+$'), re.compile(r'^\d{1,2}$')
if has('mark', 'ore'):
    for i in df.index[is_data]:
        m, o = df.at[i, 'mark'].strip(), df.at[i, 'ore'].strip()
        if m and not num_re.match(m): flag('type', i, f'mark not numeric: {m!r}')
        if o and not ore_re.match(o): flag('type', i, f'ore not 2-digit: {o!r}')
        if m and not o: flag('type', i, 'mark without ore')
else: skipped.append('type (no mark/ore)')

# leak: digits in any text column
if text_cols:
    for i in df.index[is_data]:
        for c in text_cols:
            if re.search(r'\d', df.at[i, c]): flag('leak', i, f'{c} has digit: {df.at[i, c]!r}')
else: skipped.append('leak (no text columns)')

# words_in_CD: cols C and D are duplicate gaards/brugs (numeric); flag stray words
LET = re.compile(r'[A-Za-zÆØÅæøå]')
for pos in (2, 3):
    if pos < len(cols):
        col = cols[pos]
        v = df.loc[is_data, col].str.strip(); ne = v[v != '']
        if len(ne) and ne.str.match(r'^\d+$').mean() > 0.5:
            for i in ne.index:
                if LET.search(df.at[i, col]): flag('words_in_CD', i, f'{col} (col {chr(65 + pos)}) words: {df.at[i, col]!r}')

# ditto: every do./„ owner has an antecedent
if 'eier_bruker' in cols:
    data_idx = list(df.index[is_data])
    for k, i in enumerate(data_idx):
        if df.at[i, 'eier_bruker'].strip() in DITTO:
            prev = [df.at[j, 'eier_bruker'].strip() for j in data_idx[:k]]
            if not any(p and p not in DITTO for p in prev): flag('ditto', i, 'ditto with no antecedent owner')
else: skipped.append('ditto (no eier_bruker)')

# balance: independent page-sum recompute (ore units)
if has('mark', 'ore') and is_total.any():
    tot = list(df.index[is_total])
    for a, c in zip(tot, tot[1:]):
        seg = df.iloc[a + 1:c]; sd = seg[is_data.iloc[a + 1:c].values]
        s = int(pd.to_numeric(sd['mark'], errors='coerce').fillna(0).sum()) * 100 + \
            int(pd.to_numeric(sd['ore'],  errors='coerce').fillna(0).sum())
        va = int(round(mk[a])) * 100 + int(round(oe[a])); vc = int(round(mk[c])) * 100 + int(round(oe[c]))
        if vc - va != s: flag('balance', c, f'delta {(vc - va - s)/100:.2f} (total {vc/100:.2f}, prev {va/100:.2f}, page {s/100:.2f})')
else: skipped.append('balance (no mark/ore or no totals)')

# duplicate keys
if has('gaards_no', 'brugs_no'):
    key = df['gaards_no'].str.strip() + '-' + df['brugs_no'].str.strip()
    m = is_data & df['gaards_no'].str.strip().ne('') & df['brugs_no'].str.strip().ne('')
    for i in df.index[m & key.duplicated(keep=False)]: flag('duplicate', i, f'duplicate key {key[i]}')
else: skipped.append('duplicate (no gaards_no/brugs_no)')

# empty_skyld: mark and ore both blank
if has('mark', 'ore'):
    for i in df.index[is_data]:
        if not df.at[i, 'mark'].strip() and not df.at[i, 'ore'].strip(): flag('empty_skyld', i, 'mark and ore both blank')

# symbol: any char outside the legitimate set (Latin letters, digits, normal punctuation)
PUNCT_OK = set(" .,;:()-&/'" + '"' + "\u201e\u201c\u201d\u2018\u2019")
def weird(s):
    bad = []
    for ch in s:
        if ch.isspace() or ch.isdigit() or ch in PUNCT_OK: continue
        if ord(ch) < 128 and ch.isalpha(): continue
        if ch in "\u00c6\u00d8\u00c5\u00e6\u00f8\u00e5": continue
        try:
            if 'LATIN' in unicodedata.name(ch): continue
        except ValueError: pass
        bad.append(ch)
    return bad
for i in df.index:
    for c in scan_cols:
        bd = weird(df.at[i, c])
        if bd: flag('symbol', i, f'{c} has {"".join(sorted(set(bd)))!r}')

# markup: stray angle brackets / OCR tags like <math>347031</math>
for i in df.index:
    bad = [c for c in scan_cols if '<' in df.at[i, c] or '>' in df.at[i, c]]
    if bad: flag('markup', i, f'angle brackets in {bad}')

# summary: herred/sogn footer lines
SUMMARY = re.compile(r'herredets|\bsamlede\b|\bsamlet\b', re.I)
for i in df.index:
    if SUMMARY.search(' '.join(df.at[i, c] for c in scan_cols)): flag('summary', i, 'summary/footer line')

rep = pd.DataFrame(issues, columns=['check', 'excel_row', 'gaards_no', 'brugs_no', 'detail'])
rep.to_csv(OUT, index=False, encoding='utf-8-sig')

print('columns:', cols)
if skipped: print('skipped checks:', skipped)
print(rep['check'].value_counts().to_string() if len(rep) else 'no issues')
print(f'\n{len(rep)} issues -> {OUT}')
if has('mark', 'ore'):
    nums = ''.join(df.loc[is_data, 'mark']) + ''.join(df.loc[is_data, 'ore'])
    print('digit distribution:', dict(sorted(Counter(c for c in nums if c.isdigit()).items())))

columns: ['gaards_no', 'brugs_no', 'gaards_no_raw', 'brugs_no_raw', 'gaardens_navn', 'brugets_navn', 'eier_bruker', 'mark', 'ore', 'anmerkn', 'postanstalt', 'check']
check
duplicate      2571
structure       368
words_in_CD     166
empty_skyld     129
balance         112
symbol           84
leak             50
markup           43
type             10
summary           7

3540 issues -> validation_report.csv
digit distribution: {'0': 5316, '1': 6415, '2': 4887, '3': 3810, '4': 3327, '5': 3202, '6': 2670, '7': 2648, '8': 2281, '9': 2276}


In [10]:
import pandas as pd
report = pd.read_csv(OUT)
print(f'{len(report)} issues')
report

3540 issues


,check,excel_row,gaards_no,brugs_no,detail
0,structure,418,101.0,1.0,gaards 99 -> 101
1,structure,830,1.0,1.0,gaards 179 -> 1
2,structure,997,44.0,1.0,gaards 42 -> 44
3,structure,1452,1.0,1.0,gaards 99 -> 1
4,structure,1696,70.0,2.0,gaards 68 -> 70
...,...,...,...,...,...
3535,summary,9218,NaN,20.0,summary/footer line
3536,summary,9746,97.0,1.0,summary/footer line
3537,summary,10124,NaN,6.0,summary/footer line
3538,summary,12054,NaN,5.0,summary/footer line
